In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [3]:
datasetName = 'Compas'

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from aif360.sklearn.datasets import fetch_compas
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


dataset, target = fetch_compas()
dataset = dataset.reset_index(drop=True)
target = target.reset_index(drop=True)
TARGET_COLUMN = 'two_year_recid'
dataset[TARGET_COLUMN] = target
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(target)
dataset = dataset.drop(['c_charge_desc', 'age_cat'], axis=1)
target = dataset[TARGET_COLUMN]
datasetX = dataset.drop(['two_year_recid'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = ["age", "juv_fel_count", "juv_misd_count", "juv_other_count", "priors_count"]
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.6320907617504052


In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

iea.dataset.describe()

,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree
count,6167.000000,6167.000000,6167.000000,6167.000000,6167.000000,6167.000000,6167.000000,6167.000000
mean,0.809794,34.531863,1.218907,0.059186,0.091292,0.110751,3.247446,0.356900
std,0.392495,11.726167,1.429525,0.463630,0.498067,0.470911,4.745320,0.479124
min,0.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,31.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,1.000000,42.000000,2.000000,0.000000,0.000000,0.000000,4.000000,1.000000
max,1.000000,96.000000,5.000000,20.000000,13.000000,9.000000,38.000000,1.000000


# Constraints Type Series:
1. Immutability
2. Ranges
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'sex': 'i',
        'race': 'i'
    },
    2:{
        'sex': 'i',
        'race': 'i',
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
        
    },
    3: {
        'sex': 'i',
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
        "age": 'incr',
        "juv_fel_count": 'decr',
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_imm_ranges_direct_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_imm_ranges_direct = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_imm_ranges_direct_arr.append(results_incremental_imm_ranges_direct)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_imm_ranges_direct_arr, open(f"{results_dir}/results_incremental_imm_ranges_direct_arr.pkl", "wb"))

In [8]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_imm_ranges_direct_arr = pickle.load(open(f'{results_dir}/results_incremental_imm_ranges_direct_arr.pkl', 'rb'))

# Constraints Type Series:
1. Ranges
2. Immutability
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
    },
    2:{
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
        'sex': 'i',
        'race': 'i'
    },
    3: {
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
        'sex': 'i',
        'race': 'i',
        "age": 'incr',
        "juv_fel_count": 'decr',
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_ranges_imm_dir = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_ranges_imm_incr = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_ranges_imm_dir.append(results_incremental_ranges_imm_incr)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_ranges_imm_dir, open(f"{results_dir}/results_incremental_ranges_imm_dir.pkl", "wb"))

In [9]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_ranges_imm_dir = pickle.load(open(f'{results_dir}/results_incremental_ranges_imm_incr_arr.pkl', 'rb'))

# Constraints Type Series:
1. Directionality
2. Immutability
3. Ranges

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        "age": 'incr',
        "juv_fel_count": 'decr',
    },
    2:{
        "age": 'incr',
        "juv_fel_count": 'decr',
        'sex': 'i',
        'race': 'i'
    },
    3: {
        "age": 'incr',
        "juv_fel_count": 'decr',
        'sex': 'i',
        'race': 'i',
        'prior_count': (0, 20),
        'juv_misd_count': (0, 10),
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

import time
strategy = "fix_population_update_fitness"
results_incremental_dir_im_range_arr = []
for i in range(5):
    results_incremental_dir_im_range = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_dir_im_range_arr.append(results_incremental_dir_im_range)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_dir_im_range_arr, open(f"{results_dir}/results_incremental_dir_im_range_arr.pkl", "wb"))

In [11]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_dir_im_range_arr = pickle.load(open(f'{results_dir}/results_incremental_dir_im_range_arr.pkl', 'rb'))

# Make Plots

In [ ]:
from test_utils import gather_results_sequence_of_type_constraints
import matplotlib.pyplot as plt

constraint_orders = ["I→R→D", "R→I→D", "D→I→R"]

time_dynamic_imm_ranges_direct, avg_generations_imm_ranges_direct, avg_cfes_found_imm_ranges_direct, avg_proximity_loss_imm_ranges_direct, avg_sparsity_imm_ranges_direct, avg_intermediate_imm_ranges_incr, \
time_dynamic_ranges_imm_incr, avg_generations_ranges_imm_incr, avg_cfes_found_ranges_imm_incr, avg_proximity_loss_ranges_imm_incr, avg_sparsity_ranges_imm_incr, avg_intermediate_ranges_imm_incr, \
time_dynamic_dir_im_range, avg_generations_dir_im_range, avg_cfes_found_dir_im_range, avg_proximity_loss_dir_im_range, avg_sparsity_dir_im_range, avg_intermediate_dir_im_range =\
    gather_results_sequence_of_type_constraints(iea, results_incremental_imm_ranges_direct_arr, results_incremental_ranges_imm_dir, results_incremental_dir_im_range_arr, verbose=True) 

cfe_found = [
        avg_cfes_found_imm_ranges_direct,
        avg_cfes_found_ranges_imm_incr,
        avg_cfes_found_dir_im_range
]
avg_time = [
    time_dynamic_imm_ranges_direct,
    time_dynamic_ranges_imm_incr,
    time_dynamic_dir_im_range
]
avg_weighted_l1 = [
    avg_proximity_loss_imm_ranges_direct,
    avg_proximity_loss_ranges_imm_incr,
    avg_proximity_loss_dir_im_range
]

avg_sparsity = [
    avg_sparsity_imm_ranges_direct,
    avg_sparsity_ranges_imm_incr,
    avg_sparsity_dir_im_range
]
results = {
    "cfe_found": cfe_found,
    "avg_time": avg_time,
    "avg_weighted_l1": avg_weighted_l1,
    "avg_sparsity": avg_sparsity
}

In [ ]:
table_data = pd.DataFrame(results, index=constraint_orders)
print(table_data.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrr}
\toprule
 & cfe_found & avg_time & avg_weighted_l1 & avg_sparsity \\
\midrule
I→R→D & 95.9783 & 3.3386 & 0.0384 & 0.0284 \\
R→I→D & 93.4058 & 3.1468 & 0.0514 & 0.0318 \\
D→I→R & 93.3696 & 3.1289 & 0.0514 & 0.0310 \\
\bottomrule
\end{tabular}

